In [1]:
import pandas as pd

In [2]:
#Cargo mis datos
df_train = pd.read_csv('datos_limpios_train.csv', sep=';', encoding='utf-8')

In [3]:
import numpy as np

#Creo una función para crear nuevas variables
def crear_variables(df):
    df=df.copy()
    #Almacenamiento TOTAL, para distinguir de un solo paso el tamaño total del equipo
    df['Total_Storage_GB'] = df['SSD_GB'] + df['HDD_GB'] + df['Hybrid_GB']
    #Indicador de gama y almacenamiento
    df['Tiene_SSD'] = (df['SSD_GB'] > 0).astype(int)
    df['Es_Gaming_Workstation'] = df['TypeName'].isin(['Gaming', 'Workstation']).astype(int)
    #nuevo ratios de gama alta
    df['Peso_por_Pulgada'] = df['Weight'] / df['Inches'] # Detecta ultrabooks muy ligeros y caros
    df['Ratio_SSD'] = df['SSD_GB'] / (df['Total_Storage_GB'] + 1) # % de almacenamiento rápido (+1 evita dividir por 0)
    df['Total_Pixeles'] = df['Res_Ancho'] * df['Res_Alto'] # Detecta pantallas 4K
    #Simplifico la feauture 'Product', me quedo con la familia o serie
    df['Product_Serie'] = df['Product'].astype(str).apply(lambda x: x.split()[0].split('-')[0].lower())
    #Elimino la columna 'Product' que me generaba cientos de columnas
    df = df.drop(columns=['Product'])
    return df

In [4]:
df_train_feat = crear_variables(df_train)

In [5]:
df_train_feat.head()

,Company,TypeName,Inches,Ram_GB,OpSys,Res_Ancho,Res_Alto,Touchscreen,IPS,PPI,...,Gpu_Gama,Weight,Price_euros,Total_Storage_GB,Tiene_SSD,Es_Gaming_Workstation,Peso_por_Pulgada,Ratio_SSD,Total_Pixeles,Product_Serie
0,Asus,2 in 1 Convertible,12.5,4,Chrome OS,1920,1080,1,0,176.232574,...,Intel Integrada,1.20,669.0,64,1,0,0.096000,0.984615,2073600,chromebook
1,Asus,Gaming,17.3,8,Windows 10,1920,1080,0,0,127.335675,...,AMD Estándar,3.20,1695.0,1256,1,1,0.184971,0.203660,2073600,rog
2,Lenovo,Notebook,15.6,4,Windows 10,1920,1080,0,0,141.211998,...,Intel Integrada,1.85,779.0,1000,0,0,0.118590,0.000000,2073600,v310
3,Dell,Ultrabook,13.3,16,Windows 10,3200,1800,1,0,276.053530,...,Intel Integrada,1.29,2240.0,512,1,0,0.096992,0.998051,5760000,xps
4,Lenovo,Notebook,15.6,4,Windows 7,1920,1080,0,0,141.211998,...,Intel Integrada,2.32,825.0,1000,0,0,0.148718,0.000000,2073600,b51


In [6]:
df_train_feat.info()

<class 'pandas.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Company                898 non-null    str    
 1   TypeName               898 non-null    str    
 2   Inches                 898 non-null    float64
 3   Ram_GB                 898 non-null    int64  
 4   OpSys                  898 non-null    str    
 5   Res_Ancho              898 non-null    int64  
 6   Res_Alto               898 non-null    int64  
 7   Touchscreen            898 non-null    int64  
 8   IPS                    898 non-null    int64  
 9   PPI                    898 non-null    float64
 10  Cpu_GHz                898 non-null    float64
 11  Cpu_Alto_Rendimiento   898 non-null    int64  
 12  Cpu_Gama               898 non-null    str    
 13  SSD_GB                 898 non-null    int64  
 14  HDD_GB                 898 non-null    int64  
 15  Hybrid_GB        

In [7]:
X = df_train_feat.drop(columns=['Price_euros'])
y = df_train_feat['Price_euros']

In [8]:
#OJO el df de train es toda nuestra data (train y test)
#Así que divido para entrenar al modelo en 80/20
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
#Agrupo series de productos poco frecuentes
#cuento la frec de cada serie
conteo_series = X_train['Product_Serie'].value_counts()
series_frecuentes = conteo_series[conteo_series >= 5].index
series_frecuentes

Index(['inspiron', 'ideapad', 'thinkpad', 'aspire', 'elitebook', 'probook',
       'latitude', '15', 'vostro', 'rog', 'xps', 'zenbook', 'vivobook',
       'legion', 'macbook', '250', 'zbook', 'yoga', 'tecra', 'chromebook',
       'alienware', 'satellite', 'v110', 'pavilion', 'omen', 'portege',
       'spectre', 'v310', '17', 'x541na', 'precision'],
      dtype='str', name='Product_Serie')

In [10]:
#Reemplazo los poco frecuentes por otros
X_train['Product_Serie'] = X_train['Product_Serie'].where(X_train['Product_Serie'].isin(series_frecuentes), 'Otros')
X_test['Product_Serie'] = X_test['Product_Serie'].where(X_test['Product_Serie'].isin(series_frecuentes), 'Otros')


In [11]:
print(f"Columnas categóricas únicas en Product_Serie ahora: {X_train['Product_Serie'].nunique()}")


Columnas categóricas únicas en Product_Serie ahora: 32


In [12]:
#Convierto las columnas de texto a 0s y 1s
X_train_prep = pd.get_dummies(X_train, drop_first=True)
X_test_prep = pd.get_dummies(X_test, drop_first=True)

In [13]:
#Relleno con 0 las categorías que falten en alguno de los dos conjuntos
X_train_prep, X_test_prep = X_train_prep.align(X_test_prep, join='left', axis=1, fill_value=0)

In [14]:
#Hago un Random Forest como Baseline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

#Creo y entreno el modelo baseline
rf = RandomForestRegressor(n_estimators=200, criterion='absolute_error', random_state=42)

#entreno con logaritmo en y_train, para evitar la asimetria positiva de los precios
rf.fit(X_train_prep, np.log1p(y_train))

#Predigo y revierto el logaritmo del target con expm1
y_pred = np.expm1(rf.predict(X_test_prep))

#Evalúo/Métricas
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f} €")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f} €")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")

MAE:  156.89 €
RMSE: 242.36 €
R2:   0.8649


In [15]:
#GEMINI
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Preparamos los datos (este modelo lee texto como 'category', sin get_dummies)
X_train_hgb = X_train.copy()
X_test_hgb = X_test.copy()

cat_cols = X_train_hgb.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    X_train_hgb[col] = X_train_hgb[col].astype('category')
    X_test_hgb[col] = X_test_hgb[col].astype('category')

# 2. Entrenamos optimizando el MAE con logaritmo
hgb = HistGradientBoostingRegressor(
    loss='absolute_error',
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=10,
    categorical_features='from_dtype',
    random_state=42
)

hgb.fit(X_train_hgb, np.log1p(y_train))

# 3. Predecimos y medimos el nuevo MAE
y_pred_hgb = np.expm1(hgb.predict(X_test_hgb))
print(f"MAE HistGradientBoosting: {mean_absolute_error(y_test, y_pred_hgb):.2f} €")


MAE HistGradientBoosting: 141.09 €


In [16]:
# Combinamos las predicciones de Random Forest y HistGradientBoosting
y_pred_ensamble = (0.7 * y_pred_hgb) + (0.3 * y_pred)

print(f"MAE Ensamble (70% HGB + 30% RF): {mean_absolute_error(y_test, y_pred_ensamble):.2f} €")


MAE Ensamble (70% HGB + 30% RF): 139.28 €


In [17]:
from sklearn.model_selection import KFold, cross_val_score

# Preparamos los datos limpios para evaluar en 5 bloques (folds)
X_cv = X_train_hgb.copy()
y_cv = np.log1p(y_train)

# Configuramos 5 particiones distintas
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Calculamos el MAE en cada una de las 5 pruebas
maes = []
for train_idx, val_idx in kf.split(X_cv):
    X_tr, X_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
    y_tr, y_val = y_cv.iloc[train_idx], y_train.iloc[val_idx]
    
    hgb.fit(X_tr, y_tr)
    preds = np.expm1(hgb.predict(X_val))
    maes.append(mean_absolute_error(y_val, preds))

print(f"MAE medio real (5-Fold): {np.mean(maes):.2f} €")


MAE medio real (5-Fold): 194.25 €


In [18]:
importancias = pd.Series(rf.feature_importances_, index=X_train_prep.columns)
print("\nTop 10 variables más importantes:")
print(importancias.sort_values(ascending=False).head(10))



Top 10 variables más importantes:
Ram_GB                 0.344267
Cpu_GHz                0.065188
Weight                 0.055105
Peso_por_Pulgada       0.052274
SSD_GB                 0.046126
TypeName_Notebook      0.045052
Cpu_Gama_Otro Intel    0.031642
PPI                    0.025806
Inches                 0.024750
Ratio_SSD              0.020474
dtype: float64


In [19]:
#Busco los 5 casos de X_tet con mayor diferencia entre el precio reañ y el predicho

errores = X_test.copy()
errores['Precio_Real'] = y_test
errores['Precio_Predicho'] = y_pred.round(2)
errores['Error_Abs'] = (errores['Precio_Real'] - errores['Precio_Predicho']).abs()

#Muestro los 5 peores casos con sus características clave
cols_ver = ['Company', 'TypeName', 'Ram_GB', 'SSD_GB', 'Cpu_Gama', 'Gpu_Gama', 'Precio_Real', 'Precio_Predicho', 'Error_Abs']
print(errores.sort_values(by='Error_Abs', ascending=False)[cols_ver].head(5))


    Company   TypeName  Ram_GB  SSD_GB       Cpu_Gama  \
298    Asus     Gaming      32     512  Intel Core i7   
809   Razer     Gaming      16    1000  Intel Core i7   
703      LG  Ultrabook       8     512  Intel Core i7   
836     MSI     Gaming      16     256  Intel Core i7   
450      HP     Gaming      12     256  Intel Core i7   

                    Gpu_Gama  Precio_Real  Precio_Predicho  Error_Abs  
298  Nvidia Gaming (GTX/RTX)      1279.00          2722.77    1443.77  
809  Nvidia Gaming (GTX/RTX)      3499.00          2611.77     887.23  
703          Intel Integrada      2299.00          1507.37     791.63  
836  Nvidia Gaming (GTX/RTX)      1545.64          2209.28     663.64  
450  Nvidia Gaming (GTX/RTX)      1999.00          1346.18     652.82  


In [20]:
#Cargo el test
df_test_comp = pd.read_csv('datos_limpios_test.csv', sep=';', encoding='utf-8')

In [21]:
df_test_feat = crear_variables(df_test_comp)

In [22]:
#GEMINI
# 1. Preparamos el 100% de los datos de entrenamiento (X) y el test de competición (df_test_feat)
X_full = X.copy()
X_comp = df_test_feat.drop(columns=['id']).copy()

# Calculamos las series frecuentes sobre todo el entrenamiento y filtramos ambos
conteo_full = X_full['Product_Serie'].value_counts()
series_frec_full = conteo_full[conteo_full >= 5].index

X_full['Product_Serie'] = X_full['Product_Serie'].where(X_full['Product_Serie'].isin(series_frec_full), 'Otros')
X_comp['Product_Serie'] = X_comp['Product_Serie'].where(X_comp['Product_Serie'].isin(series_frec_full), 'Otros')

# --- MODELO 1: HistGradientBoosting con el 100% de los datos ---
X_full_hgb = X_full.copy()
X_comp_hgb = X_comp.copy()

cat_cols = X_full_hgb.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    X_full_hgb[col] = X_full_hgb[col].astype('category')
    X_comp_hgb[col] = X_comp_hgb[col].astype('category')

hgb_final = HistGradientBoostingRegressor(
    loss='absolute_error',
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=10,
    categorical_features='from_dtype',
    random_state=42
)
hgb_final.fit(X_full_hgb, np.log1p(y))
pred_hgb_final = np.expm1(hgb_final.predict(X_comp_hgb))

# --- MODELO 2: Random Forest con el 100% de los datos ---
X_full_rf = pd.get_dummies(X_full, drop_first=True)
X_comp_rf = pd.get_dummies(X_comp, drop_first=True)
X_full_rf, X_comp_rf = X_full_rf.align(X_comp_rf, join='left', axis=1, fill_value=0)

rf_final = RandomForestRegressor(n_estimators=200, criterion='absolute_error', random_state=42)
rf_final.fit(X_full_rf, np.log1p(y))
pred_rf_final = np.expm1(rf_final.predict(X_comp_rf))

# --- ENSAMBLE GANADOR (70% HGB + 30% RF) ---
predicciones_finales = (0.7 * pred_hgb_final) + (0.3 * pred_rf_final)


In [23]:
#Creo el DataFrame de submission y guardarlo
submission = pd.DataFrame({
    'id': df_test_comp['id'], 
    'Price_euros': predicciones_finales
})

submission.to_csv('submission.csv', index=False)